# Load test — Zarr NWB with AIND metadata

Only dependencies: `hdmf_zarr` (brings `NWBZarrIO`) + stdlib `json` + `numpy`. No custom class import, no extension registration.

In [3]:
from hdmf_zarr import NWBZarrIO
import json
import numpy as np

write_save_file = '/root/capsule/data/test_nwb/test.nwb.zarr'
write_save_file = '/root/capsule/data/test_nwb/behavior_749472_2025-01-09_13-56-02_combined.nwb.zarr'

In [4]:
# One IO, kept open for the rest of the notebook so lazy datasets stay accessible.
io = NWBZarrIO(write_save_file, 'r')
nwb = io.read()

# AIND metadata lives at /general/aind_metadata/json_data (LabMetaData containers
# sit directly under /general/, not under /general/lab_meta_data/).
raw = io.file['general/aind_metadata/json_data'][()]
if isinstance(raw, np.ndarray):
    raw = raw.item()
if isinstance(raw, bytes):
    raw = raw.decode()
meta = json.loads(raw)

print('metadata files:', list(meta.keys()))

metadata files: ['acquisition', 'data_description', 'instrument', 'metadata.nd', 'procedures', 'subject']


In [5]:
print('acquisition subject_id:', meta['acquisition'].get('subject_id'))
print('subject genotype:', meta['subject'].get('subject_details', {}).get('genotype'))

acquisition subject_id: 749472
subject genotype: None


## Trials

In [6]:
trials_df = nwb.trials.to_dataframe()
print(f'{len(trials_df)} rows, {len(trials_df.columns)} columns')
print('ragged sample right_reward_times[0:5]:', trials_df['right_reward_times'].head(5).tolist())
trials_df.head()

460 rows, 34 columns
ragged sample right_reward_times[0:5]: [array([], dtype=float64), array([], dtype=float64), array([], dtype=float64), array([0.624]), array([0.375])]


,start_time,stop_time,trial_type,animal_response,rewarded_historyL,rewarded_historyR,goCue_start_time,reward_outcome_time,bait_left,bait_right,...,ITI_duration,delay_max,delay_min,lick_lat,trial_ind,right_reward_times,left_reward_times,choice_time_trial,auto_manual_trial,extra_reward
id,,,,,,,,,,,,,,,,,,,,,
0,97.919,102.519,CSplus,0,False,False,99.419,99.713,0,0,...,0.5,1,1,0.294,0,[],[],0.294,False,False
1,102.519,112.820,CSplus,0,False,False,109.720,110.267,0,0,...,0.5,1,1,0.547,1,[],[],0.547,False,False
2,112.820,121.566,CSplus,0,False,False,118.466,118.678,0,0,...,0.5,1,1,0.212,2,[],[],0.212,False,False
3,121.566,125.883,CSplus,1,False,True,122.783,123.206,0,0,...,0.5,1,1,0.423,3,[0.6239999999999952],[],0.423,False,False
4,125.883,130.447,CSplus,1,False,True,127.347,127.521,0,0,...,0.5,1,1,0.174,4,[0.375],[],0.174,False,False


## Units

In [10]:
if nwb.units is None:
    print('No units table found in this NWB file.')
else:
    units_df = nwb.units.to_dataframe()
    print(f'{len(units_df)} rows, {len(units_df.columns)} columns')
    print('spike counts per unit (first 5):', [len(units_df['spike_times'].iloc[i]) for i in range(5)])
    units_df.head()

No units table found in this NWB file.


## Acquisition TimeSeries

In [11]:
for name, ts in nwb.acquisition.items():
    n = len(ts.timestamps)
    print(f'{name}: {n} timestamps, unit={ts.unit}, range=[{ts.timestamps[0]}, {ts.timestamps[-1]}]')

FIP_rising_time: 69329 timestamps, unit=second, range=[83.20958228333295, 3549.1710760502815]
G-Iso_0: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G-Iso_1: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G-Iso_2: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G_0: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G_0_bright: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G_0_bright_mc: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G_0_exp: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G_0_exp_mc: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G_0_tri-exp: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G_0_tri-exp_mc: 69329 timestamps, unit=a.u., range=[83.20958228333295, 3549.1710760502815]
G_1: 69329 timestamps, unit=a.u., range=[83.209

## Processing modules

In [12]:
for pm_name, pm in nwb.processing.items():
    print(f'{pm_name}:')
    for obj_name, obj in pm.data_interfaces.items():
        print(f'  {obj_name}: {type(obj).__name__}')

In [13]:
nwb.lab_meta_data

{'aind_metadata': aind_metadata abc.AindMetadata at 0x140270968242448
 Fields:
   json_data: {"acquisition": {"acquisition_end_time": "2025-01-09T14:51:55.917000-08:00", "acquisition_start_time": "2025-01-09T13:56:02-08:00", "acquisition_type": "Fiber photometry recording", "calibrations": [], "coordinate_system": null, "data_streams": [{"active_devices": ["470nm LED", "415nm LED", "565nm LED", "Fiber optic patch cord", "Green CMOS", "Red CMOS", "Pupil camera assembly", "Tongue camera assembly", "Lick spout assembly", "Speaker", "Arduino", "Fiber 0", "Fiber 1", "Fiber 2", "Fiber 3", "Patch Cord A", "Patch Cord B", "Patch Cord C", "Patch Cord D"], "code": null, "configurations": [{"device_name": "470nm LED", "object_type": "Light emitting diode config", "power": 60, "power_unit": "microwatt"}, {"device_name": "415nm LED", "object_type": "Light emitting diode config", "power": 40, "power_unit": "microwatt"}, {"device_name": "565nm LED", "object_type": "Light emitting diode config", "power": 0, "power_unit": "microwatt"}, {"channels": [{"additional_device_names": null, "channel_name": "Fiber channel", "detector": {"compression": null, "device_name": "Green CMOS", "exposure_time": 0, "exposure_time_unit": "second", "object_type": "Detector config", "trigger_type": "Internal"}, "emission_filters": null, "emission_wavelength": null, "emission_wavelength_unit": null, "excitation_filters": null, "intended_measurement": null, "light_sources": [], "object_type": "Channel", "variable_power": false}, {"additional_device_names": null, "channel_name": "Fiber channel", "detector": {"compression": null, "device_name": "Red CMOS", "exposure_time": 0, "exposure_time_unit": "second", "object_type": "Detector config", "trigger_type": "Internal"}, "emission_filters": null, "emission_wavelength": null, "emission_wavelength_unit": null, "excitation_filters": null, "intended_measurement": null, "light_sources": [], "object_type": "Channel", "variable_power": false}], "device_name": "Fiber optic patch cord", "object_type": "Patch cord config"}], "connections": [{"object_type": "Connection", "send_and_receive": true, "source_device": "Fiber 0", "source_port": null, "target_device": "Patch Cord A", "target_port": null}, {"object_type": "Connection", "send_and_receive": true, "source_device": "Fiber 1", "source_port": null, "target_device": "Patch Cord B", "target_port": null}, {"object_type": "Connection", "send_and_receive": true, "source_device": "Fiber 2", "source_port": null, "target_device": "Patch Cord C", "target_port": null}, {"object_type": "Connection", "send_and_receive": true, "source_device": "Fiber 3", "source_port": null, "target_device": "Patch Cord D", "target_port": null}], "modalities": [{"abbreviation": "behavior", "name": "Behavior"}, {"abbreviation": "fib", "name": "Fiber photometry"}], "notes": "LED power measured at patch cable end; fib mode: Axon", "object_type": "Data stream", "stream_end_time": "2025-01-09T14:51:55.917000-08:00", "stream_start_time": "2025-01-09T13:56:02-08:00"}], "describedBy": "https://raw.githubusercontent.com/AllenNeuralDynamics/aind-data-schema/main/src/aind_data_schema/core/acquisition.py", "ethics_review_id": ["2115"], "experimenters": ["Sue Su"], "instrument_id": "FIP_Sue", "maintenance": [], "notes": "Legacy fiber photometry recording session for subject 749472. Note: Actual power and exposure duration values are not available in the legacy data.", "object_type": "Acquisition", "protocol_id": null, "schema_version": "2.0.35", "specimen_id": null, "stimulus_epochs": [{"active_devices": ["Speaker"], "code": null, "configurations": [], "curriculum_status": null, "notes": "Behavioral foraging task during fiber photometry recording", "object_type": "Stimulus epoch", "performance_metrics": {"object_type": "Performance metrics", "output_parameters": {}, "reward_consumed_during_epoch": "561.0", "reward_consumed_unit": "microliter", "trials_finished": 460, "trials_rewarded": 187, "trials_total": 460}, "stimulu